# Group36 (Manav Gupta) COMP90042 Project 2026.ipynb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q rank_bm25 sentence-transformers transformers accelerate
!pip install -q sympy==1.13.3

import subprocess, sys
import torch
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-gpu'])
except:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json, os, pickle, re, random, gc, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
import faiss
from collections import defaultdict, Counter
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

LABEL2ID = {'SUPPORTS': 0, 'REFUTES': 1, 'NOT_ENOUGH_INFO': 2, 'DISPUTED': 3}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

Device: cuda


In [ ]:
FOLDER_NAME = 'COMP90042'
DATA_DIR = f'/content/drive/MyDrive/{FOLDER_NAME}/'
CKPT_DIR = DATA_DIR + 'checkpoints/'
os.makedirs(CKPT_DIR, exist_ok=True)

BM25_PATH = CKPT_DIR + 'bm25_index.pkl'
FAISS_PATH = CKPT_DIR + 'evidence_faiss.index'
IDS_PATH = CKPT_DIR + 'evidence_ids.pkl'
EMBEDDINGS_PATH = CKPT_DIR + 'evidence_embeddings.npy'
TFIDF_PATH = CKPT_DIR + 'tfidf_vectorizer.pkl'
TFIDF_MATRIX_PATH = CKPT_DIR + 'tfidf_matrix.pkl'
RETRIEVAL_PATH = CKPT_DIR + 'retrieval_scores.json'
BILSTM_PATH = CKPT_DIR + 'best_bilstm.pt'
BERT_DIR = CKPT_DIR + 'best_bert/'
ROBERTA_DIR = CKPT_DIR + 'best_roberta/'
ROBERTA_FINAL_DIR = CKPT_DIR + 'best_roberta_final/'
TRAIN_SAMPLES_PATH = CKPT_DIR + 'train_samples_gold.pkl'
DEV_SAMPLES_PATH = CKPT_DIR + 'dev_samples_gold.pkl'
ALL_SAMPLES_PATH = CKPT_DIR + 'all_samples_gold.pkl'
EMBEDDINGS_DIR = CKPT_DIR + 'embedding_chunks/'

for d in [BERT_DIR, ROBERTA_DIR, ROBERTA_FINAL_DIR, EMBEDDINGS_DIR]:
    os.makedirs(d, exist_ok=True)

# 1.DataSet Processing

In [ ]:
def load_json(path):
    with open(path) as f:
        return json.load(f)

train_claims = load_json(DATA_DIR + 'train-claims.json')
dev_claims = load_json(DATA_DIR + 'dev-claims.json')
test_claims = load_json(DATA_DIR + 'test-claims-unlabelled.json')
evidence_db = load_json(DATA_DIR + 'evidence.json')

evidence_ids = list(evidence_db.keys())
evidence_texts = [evidence_db[eid] for eid in evidence_ids]

print(f'train={len(train_claims)}, dev={len(dev_claims)}, test={len(test_claims)}, evidence={len(evidence_db)}')

train=1228, dev=154, test=153, evidence=1208827


In [ ]:
def tokenize_bm25(text):
    return re.sub(r'[^\w\s]', ' ', text.lower()).split()

if os.path.exists(BM25_PATH):
    with open(BM25_PATH, 'rb') as f:
        bm25 = pickle.load(f)
else:
    bm25 = BM25Okapi([tokenize_bm25(t) for t in evidence_texts])
    with open(BM25_PATH, 'wb') as f:
        pickle.dump(bm25, f)

def bm25_retrieve(query, top_k=100):
    scores = bm25.get_scores(tokenize_bm25(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(evidence_ids[i], float(scores[i])) for i in top_idx]

sample_claim = list(train_claims.values())[0]['claim_text']

In [ ]:
sbert = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
sbert_encode_device = 'cuda' if torch.cuda.is_available() else 'cpu'
ENCODE_BATCH = 64 if sbert_encode_device == 'cuda' else 32
EMB_DIM = 768
CHUNK_SIZE = 10_000
total_docs = len(evidence_texts)
n_chunks = (total_docs + CHUNK_SIZE - 1) // CHUNK_SIZE

if os.path.exists(FAISS_PATH) and os.path.exists(IDS_PATH) and os.path.exists(EMBEDDINGS_PATH):
    faiss_index = faiss.read_index(FAISS_PATH)
    with open(IDS_PATH, 'rb') as f:
        evidence_ids = pickle.load(f)
    print(f'FAISS loaded: {faiss_index.ntotal:,} vectors')
else:
    faiss_index = faiss.IndexFlatIP(EMB_DIM)
    run_start = time.time()
    for chunk_i, start_idx in enumerate(range(0, total_docs, CHUNK_SIZE)):
        end_idx = min(start_idx + CHUNK_SIZE, total_docs)
        chunk_path = os.path.join(EMBEDDINGS_DIR, f'chunk_{start_idx}_{end_idx}.npy')
        if os.path.exists(chunk_path):
            chunk_embs = np.load(chunk_path).astype(np.float32)
            faiss_index.add(chunk_embs)
            del chunk_embs; gc.collect()
            continue
        texts_chunk = evidence_texts[start_idx:end_idx]
        chunk_embs = sbert.encode(texts_chunk, batch_size=ENCODE_BATCH, show_progress_bar=False,
            convert_to_numpy=True, normalize_embeddings=True, device=sbert_encode_device).astype(np.float32)
        np.save(chunk_path, chunk_embs)
        faiss_index.add(chunk_embs)
        print(f'chunk {chunk_i+1}/{n_chunks} done, {faiss_index.ntotal:,} vectors')
        del chunk_embs, texts_chunk; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    chunk_files = sorted([f for f in os.listdir(EMBEDDINGS_DIR) if f.endswith('.npy')],
        key=lambda x: int(x.split('_')[1]))
    parts = [np.load(os.path.join(EMBEDDINGS_DIR, f)) for f in chunk_files]
    evidence_embeddings = np.vstack(parts).astype(np.float32)
    del parts; gc.collect()
    np.save(EMBEDDINGS_PATH, evidence_embeddings)
    faiss.write_index(faiss_index, FAISS_PATH)
    with open(IDS_PATH, 'wb') as f:
        pickle.dump(evidence_ids, f)
    print(f'done in {(time.time()-run_start)/60:.1f} mins')

assert faiss_index.ntotal == len(evidence_ids)

def dense_retrieve(query, top_k=100):
    q_emb = sbert.encode([query], convert_to_numpy=True, normalize_embeddings=True,
        device=sbert_encode_device).astype(np.float32)
    scores, indices = faiss_index.search(q_emb, top_k)
    return [(evidence_ids[idx], float(scores[0][i])) for i, idx in enumerate(indices[0]) if idx != -1]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS loaded: 1,208,827 vectors


In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    scores = defaultdict(float)
    for ranked in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked, start=1):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

STOPWORDS = {
    'the','a','an','is','are','was','were','of','in','on','at','to','for',
    'and','or','but','it','this','that','by','with','as','from','be','has',
    'have','had','its','which','who','not','no','may','can','will','would',
    'also','more','than','their','they','been','being','such','some','about'
}

def expand_query(query, top_k_bm25=3):
    top_docs = bm25_retrieve(query, top_k=top_k_bm25)
    expansion_text = ' '.join([evidence_db[eid] for eid, _ in top_docs if eid in evidence_db])
    freq = Counter(tokenize_bm25(expansion_text))
    query_words = set(tokenize_bm25(query))
    terms = [w for w, _ in freq.most_common(30)
             if w not in query_words and w not in STOPWORDS and len(w) > 3][:5]
    return query + ' ' + ' '.join(terms)

if os.path.exists(TFIDF_PATH) and os.path.exists(TFIDF_MATRIX_PATH):
    with open(TFIDF_PATH, 'rb') as f: tfidf_vectorizer = pickle.load(f)
    with open(TFIDF_MATRIX_PATH, 'rb') as f: tfidf_matrix = pickle.load(f)
    print(f'TF-IDF loaded: {tfidf_matrix.shape}')
else:
    tfidf_vectorizer = TfidfVectorizer(max_features=100_000, sublinear_tf=True, ngram_range=(1, 2), min_df=2)
    tfidf_matrix = tfidf_vectorizer.fit_transform(evidence_texts)
    with open(TFIDF_PATH, 'wb') as f: pickle.dump(tfidf_vectorizer, f)
    with open(TFIDF_MATRIX_PATH, 'wb') as f: pickle.dump(tfidf_matrix, f)

def tfidf_retrieve(query, top_k=100):
    q_vec = tfidf_vectorizer.transform([query])
    sims = sk_cosine(q_vec, tfidf_matrix).flatten()
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(evidence_ids[i], float(sims[i])) for i in top_idx]

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', max_length=512,
    device='cuda' if torch.cuda.is_available() else 'cpu')

def hybrid_retrieve(query, top_k=5, rerank_pool=25):
    expanded = expand_query(query)
    fused = reciprocal_rank_fusion([
        bm25_retrieve(query, top_k=100),
        bm25_retrieve(expanded, top_k=100),
        dense_retrieve(query, top_k=100),
        tfidf_retrieve(query, top_k=100),
    ])
    pool_ids = [eid for eid, _ in fused[:rerank_pool]]
    pairs = [[query, evidence_db[eid]] for eid in pool_ids if eid in evidence_db]
    ce_scores = cross_encoder.predict(pairs, batch_size=64, show_progress_bar=False)
    ranked = sorted(zip(pool_ids, ce_scores), key=lambda x: x[1], reverse=True)
    return [eid for eid, _ in ranked[:top_k]]

TF-IDF loaded: (1208827, 100000)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
def evidence_fscore(pred_ids, gold_ids):
    pred, gold = set(pred_ids), set(gold_ids)
    if not pred or not gold: return 0.0, 0.0, 0.0
    tp = len(pred & gold)
    p, r = tp / len(pred), tp / len(gold)
    return p, r, (2*p*r/(p+r) if (p+r) > 0 else 0.0)

def recall_at_k(ks=(5, 10, 20, 50)):
    results = {k: [] for k in ks}
    for cd in dev_claims.values():
        gold = set(cd['evidences'])
        fused = reciprocal_rank_fusion([
            bm25_retrieve(cd['claim_text'], top_k=100),
            bm25_retrieve(expand_query(cd['claim_text']), top_k=100),
            dense_retrieve(cd['claim_text'], top_k=100),
            tfidf_retrieve(cd['claim_text'], top_k=100),
        ])
        ordered = [eid for eid, _ in fused]
        for k in ks:
            results[k].append(int(bool(gold & set(ordered[:k]))))
    return {k: float(np.mean(v)) for k, v in results.items()}

if os.path.exists(RETRIEVAL_PATH):
    with open(RETRIEVAL_PATH) as f:
        ret_scores = json.load(f)
    f_bm25 = ret_scores['bm25']
    f_dense = ret_scores['dense']
    f_tfidf = ret_scores.get('tfidf', 0.0)
    f_hybrid = ret_scores['hybrid']
    recall_stats = ret_scores.get('recall_at_k', {})
else:
    def eval_retriever(fn, top_k=5):
        return float(np.mean([
            evidence_fscore(fn(cd['claim_text'], top_k=top_k), cd['evidences'])[2]
            for cd in dev_claims.values()
        ]))

    f_bm25 = eval_retriever(lambda q, top_k: [e for e,_ in bm25_retrieve(q, top_k)])
    print(f'BM25: {f_bm25:.4f}')
    f_dense = eval_retriever(lambda q, top_k: [e for e,_ in dense_retrieve(q, top_k)])
    print(f'Dense: {f_dense:.4f}')
    f_tfidf = eval_retriever(lambda q, top_k: [e for e,_ in tfidf_retrieve(q, top_k)])
    print(f'TF-IDF: {f_tfidf:.4f}')
    f_hybrid = eval_retriever(hybrid_retrieve)
    print(f'Hybrid: {f_hybrid:.4f}')
    recall_stats = recall_at_k()

    with open(RETRIEVAL_PATH, 'w') as f:
        json.dump({'bm25': f_bm25, 'dense': f_dense, 'tfidf': f_tfidf,
                   'hybrid': f_hybrid, 'recall_at_k': recall_stats}, f)

print(f'BM25={f_bm25:.4f}  Dense={f_dense:.4f}  TF-IDF={f_tfidf:.4f}  Hybrid={f_hybrid:.4f}')
if recall_stats:
    for k, r in sorted(recall_stats.items(), key=lambda x: int(x[0])):
        print(f'Recall@{k}: {r:.4f}')

BM25=0.1053  Dense=0.1507  TF-IDF=0.0569  Hybrid=0.1850
Recall@5: 0.3896
Recall@10: 0.4610
Recall@20: 0.5584
Recall@50: 0.6818


# 2.Model Implementation

In [ ]:
def build_text(claim_text, ev_ids, max_ev=3):
    parts = [claim_text] + [evidence_db[e] for e in ev_ids[:max_ev] if e in evidence_db]
    return ' [SEP] '.join(parts)

if os.path.exists(TRAIN_SAMPLES_PATH) and os.path.exists(DEV_SAMPLES_PATH):
    with open(TRAIN_SAMPLES_PATH, 'rb') as f: train_samples = pickle.load(f)
    with open(DEV_SAMPLES_PATH, 'rb') as f: dev_samples = pickle.load(f)
else:
    train_samples = [{'id': cid, 'text': build_text(cd['claim_text'], cd['evidences'][:3]),
                      'label': LABEL2ID[cd['claim_label']]} for cid, cd in train_claims.items()]
    dev_samples = [{'id': cid, 'text': build_text(cd['claim_text'], cd['evidences'][:3]),
                    'label': LABEL2ID[cd['claim_label']]} for cid, cd in dev_claims.items()]
    with open(TRAIN_SAMPLES_PATH, 'wb') as f: pickle.dump(train_samples, f)
    with open(DEV_SAMPLES_PATH, 'wb') as f: pickle.dump(dev_samples, f)

label_dist = Counter(s['label'] for s in train_samples)
n_classes = len(LABEL2ID)
total = sum(label_dist.values())
class_weights = torch.tensor(
    [total / (n_classes * label_dist[i]) for i in range(n_classes)], dtype=torch.float32
).to(DEVICE)
print(f'class weights: {[round(w, 3) for w in class_weights.tolist()]}')


class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.weight = weight

    def forward(self, logits, targets):
        n_cls = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            smooth = torch.full_like(log_probs, self.smoothing / (n_cls - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        loss = -(smooth * log_probs).sum(dim=-1)
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()


class TransformerDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=256):
        self.labels = [s['label'] for s in samples]
        self.encodings = tokenizer([s['text'] for s in samples],
            truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return ({k: v[idx] for k, v in self.encodings.items()},
                torch.tensor(self.labels[idx], dtype=torch.long))


class BiLSTMDataset(Dataset):
    def __init__(self, samples, encoder):
        self.labels = [s['label'] for s in samples]
        self.embs = encoder.encode([s['text'] for s in samples], batch_size=32,
            show_progress_bar=True, convert_to_numpy=True, device=sbert_encode_device)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (torch.tensor(self.embs[idx], dtype=torch.float32),
                torch.tensor(self.labels[idx], dtype=torch.long))


def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        labels = labels.to(DEVICE)
        optimizer.zero_grad()
        if isinstance(inputs, dict):
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            logits = model(**inputs).logits
        else:
            logits = model(inputs.to(DEVICE))
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        total_loss += loss.item()
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, criterion, is_transformer=True):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            labels = labels.to(DEVICE)
            if is_transformer:
                inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
                logits = model(**inputs).logits
            else:
                logits = model(inputs.to(DEVICE))
            loss = criterion(logits, labels)
            total_loss += loss.item()
            preds = logits.argmax(-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    return total_loss / len(loader), correct / total, all_preds, all_labels

class weights: [0.592, 1.543, 0.795, 2.476]


In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.4, chunk_size=64):
        super().__init__()
        self.chunk_size = chunk_size
        self.seq_len = input_dim // chunk_size
        self.bilstm = nn.LSTM(input_size=chunk_size, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        B = x.size(0)
        x = x.view(B, self.seq_len, self.chunk_size)
        out, _ = self.bilstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        return self.classifier(self.drop(ctx))


bilstm_criterion = LabelSmoothingCrossEntropy(smoothing=0.1, weight=None)

if os.path.exists(BILSTM_PATH):
    bilstm = BiLSTMClassifier(input_dim=768).to(DEVICE)
    bilstm.load_state_dict(torch.load(BILSTM_PATH, map_location=DEVICE))
    bilstm_dev_ds = BiLSTMDataset(dev_samples, sbert)
    bilstm_dev_loader = DataLoader(bilstm_dev_ds, batch_size=64)
    _, best_bilstm_acc, _, _ = eval_epoch(bilstm, bilstm_dev_loader, bilstm_criterion, is_transformer=False)
    print(f'BiLSTM dev accuracy: {best_bilstm_acc:.4f}')
else:
    bilstm_train_ds = BiLSTMDataset(train_samples, sbert)
    bilstm_dev_ds = BiLSTMDataset(dev_samples, sbert)
    bilstm_train_loader = DataLoader(bilstm_train_ds, batch_size=32, shuffle=True)
    bilstm_dev_loader = DataLoader(bilstm_dev_ds, batch_size=64)

    bilstm = BiLSTMClassifier(input_dim=768).to(DEVICE)
    opt_bl = AdamW(bilstm.parameters(), lr=5e-4, weight_decay=1e-4)
    best_bilstm_acc = 0
    patience, no_improve = 5, 0

    for epoch in range(15):
        tr_loss, tr_acc = train_epoch(bilstm, bilstm_train_loader, opt_bl, None, bilstm_criterion)
        _, dv_acc, _, _ = eval_epoch(bilstm, bilstm_dev_loader, bilstm_criterion, is_transformer=False)
        if dv_acc > best_bilstm_acc:
            best_bilstm_acc = dv_acc
            torch.save(bilstm.state_dict(), BILSTM_PATH)
            no_improve = 0
        else:
            no_improve += 1
        print(f'Epoch {epoch+1}/15 | train_acc={tr_acc:.4f} | dev_acc={dv_acc:.4f}')
        if no_improve >= patience:
            print('early stopping')
            break

    bilstm.load_state_dict(torch.load(BILSTM_PATH, map_location=DEVICE))
    print(f'best bilstm dev acc: {best_bilstm_acc:.4f}')

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

BiLSTM dev accuracy: 0.4416


In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_criterion = LabelSmoothingCrossEntropy(smoothing=0.1, weight=class_weights)

if os.path.exists(os.path.join(BERT_DIR, 'config.json')):
    bert = AutoModelForSequenceClassification.from_pretrained(
        BERT_DIR, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID).to(DEVICE)
    bert_dev_ds = TransformerDataset(dev_samples, bert_tokenizer)
    bert_dev_loader = DataLoader(bert_dev_ds, batch_size=16)
    _, best_bert_acc, _, _ = eval_epoch(bert, bert_dev_loader, bert_criterion)
    print(f'BERT dev acc: {best_bert_acc:.4f}')
else:
    bert_train_ds = TransformerDataset(train_samples, bert_tokenizer)
    bert_dev_ds = TransformerDataset(dev_samples, bert_tokenizer)
    bert_train_loader = DataLoader(bert_train_ds, batch_size=16, shuffle=True)
    bert_dev_loader = DataLoader(bert_dev_ds, batch_size=16)

    bert = AutoModelForSequenceClassification.from_pretrained(
        'bert-base-uncased', num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID).to(DEVICE)

    BERT_EPOCHS = 15
    opt_bert = AdamW(bert.parameters(), lr=1e-5, weight_decay=0.05)
    sched_bert = get_linear_schedule_with_warmup(opt_bert,
        num_warmup_steps=int(0.1 * len(bert_train_loader) * BERT_EPOCHS),
        num_training_steps=len(bert_train_loader) * BERT_EPOCHS)
    best_bert_acc = 0
    patience, no_improve = 3, 0

    for epoch in range(BERT_EPOCHS):
        tr_loss, tr_acc = train_epoch(bert, bert_train_loader, opt_bert, sched_bert, bert_criterion)
        _, dv_acc, _, _ = eval_epoch(bert, bert_dev_loader, bert_criterion)
        if dv_acc > best_bert_acc:
            best_bert_acc = dv_acc
            bert.save_pretrained(BERT_DIR)
            bert_tokenizer.save_pretrained(BERT_DIR)
            no_improve = 0
        else:
            no_improve += 1
        print(f'Epoch {epoch+1}/{BERT_EPOCHS} | loss={tr_loss:.4f} acc={tr_acc:.4f} | dev={dv_acc:.4f}')
        if no_improve >= patience:
            print('early stopping')
            break

    bert = AutoModelForSequenceClassification.from_pretrained(
        BERT_DIR, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID).to(DEVICE)
    print(f'best bert dev acc: {best_bert_acc:.4f}')

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

BERT dev acc: 0.5584


In [ ]:
roberta_tokenizer = AutoTokenizer.from_pretrained('roberta-base')
roberta_criterion = LabelSmoothingCrossEntropy(smoothing=0.1, weight=class_weights)

if os.path.exists(os.path.join(ROBERTA_DIR, 'config.json')):
    roberta = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_DIR, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID).to(DEVICE)
    rob_dev_ds = TransformerDataset(dev_samples, roberta_tokenizer)
    rob_dev_loader = DataLoader(rob_dev_ds, batch_size=16)
    _, best_roberta_acc, _, _ = eval_epoch(roberta, rob_dev_loader, roberta_criterion)
    print(f'RoBERTa dev acc: {best_roberta_acc:.4f}')
else:
    rob_train_ds = TransformerDataset(train_samples, roberta_tokenizer)
    rob_dev_ds = TransformerDataset(dev_samples, roberta_tokenizer)
    rob_train_loader = DataLoader(rob_train_ds, batch_size=16, shuffle=True)
    rob_dev_loader = DataLoader(rob_dev_ds, batch_size=16)

    roberta = AutoModelForSequenceClassification.from_pretrained(
        'roberta-base', num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID).to(DEVICE)

    ROB_EPOCHS = 15
    opt_rob = AdamW(roberta.parameters(), lr=1e-5, weight_decay=0.05)
    sched_rob = get_linear_schedule_with_warmup(opt_rob,
        num_warmup_steps=int(0.1 * len(rob_train_loader) * ROB_EPOCHS),
        num_training_steps=len(rob_train_loader) * ROB_EPOCHS)
    best_roberta_acc = 0
    patience, no_improve = 3, 0

    for epoch in range(ROB_EPOCHS):
        tr_loss, tr_acc = train_epoch(roberta, rob_train_loader, opt_rob, sched_rob, roberta_criterion)
        _, dv_acc, _, _ = eval_epoch(roberta, rob_dev_loader, roberta_criterion)
        if dv_acc > best_roberta_acc:
            best_roberta_acc = dv_acc
            roberta.save_pretrained(ROBERTA_DIR)
            roberta_tokenizer.save_pretrained(ROBERTA_DIR)
            no_improve = 0
        else:
            no_improve += 1
        print(f'Epoch {epoch+1}/{ROB_EPOCHS} | loss={tr_loss:.4f} acc={tr_acc:.4f} | dev={dv_acc:.4f}')
        if no_improve >= patience:
            print('early stopping')
            break

    roberta = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_DIR, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID).to(DEVICE)
    print(f'best roberta dev acc: {best_roberta_acc:.4f}')

primary_model = roberta
primary_tokenizer = roberta_tokenizer
primary_name = 'RoBERTa-base'

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RoBERTa dev acc: 0.7143


# 3.Testing and Evaluation

In [ ]:
print(f'BiLSTM:   {best_bilstm_acc:.4f}')
print(f'BERT:     {best_bert_acc:.4f}')
print(f'RoBERTa:  {best_roberta_acc:.4f}')

primary_model.eval()
dv_ds = TransformerDataset(dev_samples, primary_tokenizer)
dv_ldr = DataLoader(dv_ds, batch_size=16)
_, _, preds, labels = eval_epoch(primary_model, dv_ldr, roberta_criterion)
print(classification_report(labels, preds,
    target_names=['SUPPORTS', 'REFUTES', 'NOT_ENOUGH_INFO', 'DISPUTED'], digits=4))

BiLSTM:   0.4416
BERT:     0.5584
RoBERTa:  0.7143
                 precision    recall  f1-score   support

       SUPPORTS     0.7317    0.8824    0.8000        68
        REFUTES     0.6667    0.5185    0.5833        27
NOT_ENOUGH_INFO     0.7692    0.7317    0.7500        41
       DISPUTED     0.5000    0.3333    0.4000        18

       accuracy                         0.7143       154
      macro avg     0.6669    0.6165    0.6333       154
   weighted avg     0.7032    0.7143    0.7019       154



In [ ]:
def run_pipeline(claims_dict, model, tokenizer, top_k=5, batch_size=16):
    model.eval()
    claim_ids = list(claims_dict.keys())
    retrieved = {}
    for i, cid in enumerate(claim_ids):
        retrieved[cid] = hybrid_retrieve(claims_dict[cid]['claim_text'], top_k=top_k)
        if (i + 1) % 10 == 0 or (i + 1) == len(claim_ids):
            print(f'Retrieved {i+1}/{len(claim_ids)}')

    samples = [{'id': cid, 'text': build_text(claims_dict[cid]['claim_text'], retrieved[cid]),
                'label': 0} for cid in claim_ids]
    ds = TransformerDataset(samples, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size)
    all_preds = []
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            all_preds.extend(model(**inputs).logits.argmax(-1).cpu().tolist())

    return {cid: {'claim_text': claims_dict[cid]['claim_text'],
                  'claim_label': ID2LABEL[p], 'evidences': retrieved[cid]}
            for cid, p in zip(claim_ids, all_preds)}


def score_predictions(predictions, ground_truth):
    f_scores, correct, total = [], 0, 0
    for cid, gt in ground_truth.items():
        if cid not in predictions: continue
        pred = predictions[cid]
        _, _, f1 = evidence_fscore(pred['evidences'], gt['evidences'])
        f_scores.append(f1)
        correct += int(pred['claim_label'] == gt['claim_label'])
        total += 1
    F = float(np.mean(f_scores))
    A = correct / total
    H = 2*F*A/(F+A) if (F+A) > 0 else 0
    return {'F': F, 'A': A, 'H': H}


dev_pred_ckpt = CKPT_DIR + 'dev_predictions.json'
if os.path.exists(dev_pred_ckpt):
    with open(dev_pred_ckpt) as f:
        dev_predictions = json.load(f)
    metrics = score_predictions(dev_predictions, dev_claims)
else:
    dev_predictions = run_pipeline(dev_claims, primary_model, primary_tokenizer)
    metrics = score_predictions(dev_predictions, dev_claims)
    with open(dev_pred_ckpt, 'w') as f:
        json.dump(dev_predictions, f, indent=2)

print(f'F={metrics["F"]:.4f}  A={metrics["A"]:.4f}  H={metrics["H"]:.4f}')

F=0.1682  A=0.5065  H=0.2525


In [ ]:
def pipeline_with_retriever(claims_dict, retriever_fn, model, tokenizer, top_k=5, batch_size=16):
    model.eval()
    claim_ids = list(claims_dict.keys())
    retrieved = {}
    for i, cid in enumerate(claim_ids):
        retrieved[cid] = retriever_fn(claims_dict[cid]['claim_text'], top_k=top_k)
    samples = [{'id': cid, 'text': build_text(claims_dict[cid]['claim_text'], retrieved[cid]),
                'label': 0} for cid in claim_ids]
    ds = TransformerDataset(samples, tokenizer)
    loader = DataLoader(ds, batch_size=batch_size)
    preds = []
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            preds.extend(model(**inputs).logits.argmax(-1).cpu().tolist())
    predictions = {cid: {'claim_label': ID2LABEL[p], 'evidences': retrieved[cid]}
                   for cid, p in zip(claim_ids, preds)}
    return score_predictions(predictions, claims_dict)


ablation_path = CKPT_DIR + 'ablation_results.json'
if os.path.exists(ablation_path):
    with open(ablation_path) as f:
        res = json.load(f)
    m_bm25, m_dense, m_tfidf, m_hybrid = res['BM25'], res['Dense'], res['TF-IDF'], res['Hybrid']
else:
    m_bm25 = pipeline_with_retriever(dev_claims, lambda q, top_k: [e for e,_ in bm25_retrieve(q, top_k)],
        primary_model, primary_tokenizer)
    m_dense = pipeline_with_retriever(dev_claims, lambda q, top_k: [e for e,_ in dense_retrieve(q, top_k)],
        primary_model, primary_tokenizer)
    m_tfidf = pipeline_with_retriever(dev_claims, lambda q, top_k: [e for e,_ in tfidf_retrieve(q, top_k)],
        primary_model, primary_tokenizer)
    m_hybrid = metrics
    with open(ablation_path, 'w') as f:
        json.dump({'BM25': m_bm25, 'Dense': m_dense, 'TF-IDF': m_tfidf, 'Hybrid': m_hybrid}, f, indent=2)

print(f'BM25:    F={m_bm25["F"]:.4f}  A={m_bm25["A"]:.4f}  H={m_bm25["H"]:.4f}')
print(f'Dense:   F={m_dense["F"]:.4f}  A={m_dense["A"]:.4f}  H={m_dense["H"]:.4f}')
print(f'TF-IDF:  F={m_tfidf["F"]:.4f}  A={m_tfidf["A"]:.4f}  H={m_tfidf["H"]:.4f}')
print(f'Hybrid:  F={m_hybrid["F"]:.4f}  A={m_hybrid["A"]:.4f}  H={m_hybrid["H"]:.4f}')

BM25:    F=0.1053  A=0.4740  H=0.1723
Dense:   F=0.1507  A=0.5000  H=0.2316
TF-IDF:  F=0.0569  A=0.4545  H=0.1012
Hybrid:  F=0.1682  A=0.5065  H=0.2525


In [ ]:
from collections import defaultdict as ddict

label_names = ['SUPPORTS', 'REFUTES', 'NOT_ENOUGH_INFO', 'DISPUTED']
y_true = [LABEL2ID[dev_claims[cid]['claim_label']] for cid in dev_predictions]
y_pred = [LABEL2ID[dev_predictions[cid]['claim_label']] for cid in dev_predictions]

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(label_names, rotation=30, ha='right')
ax.set_yticklabels(label_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for i in range(4):
    for j in range(4):
        ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im); plt.tight_layout()
plt.savefig(CKPT_DIR + 'confusion_matrix.png', dpi=150)
plt.show()

errors = [(cid, dev_claims[cid], dev_predictions[cid]) for cid in dev_predictions
          if dev_predictions[cid]['claim_label'] != dev_claims[cid]['claim_label']]
print(f'errors: {len(errors)}/{len(dev_predictions)}')

confusion_pairs = ddict(list)
for cid, gold, pred in errors:
    confusion_pairs[f"{gold['claim_label']} -> {pred['claim_label']}"].append((cid, gold, pred))
for pair, cases in sorted(confusion_pairs.items(), key=lambda x: -len(x[1])):
    print(f'  {pair}: {len(cases)}')

rr, rw, mr, mw = 0, 0, 0, 0
for cid, gold in dev_claims.items():
    if cid not in dev_predictions: continue
    pred = dev_predictions[cid]
    ret_hit = bool(set(gold['evidences']) & set(pred['evidences']))
    cls_hit = pred['claim_label'] == gold['claim_label']
    if ret_hit and cls_hit: rr += 1
    elif ret_hit: rw += 1
    elif cls_hit: mr += 1
    else: mw += 1
print(f'ret_hit+cls_ok={rr}  ret_hit+cls_fail={rw}  ret_miss+cls_ok={mr}  ret_miss+cls_fail={mw}')

for cid, gold, pred in errors[:5]:
    print(f'\nclaim: {gold["claim_text"][:100]}')
    print(f'true={gold["claim_label"]}  pred={pred["claim_label"]}')
    overlap = set(pred['evidences']) & set(gold['evidences'])
    print(f'overlap: {overlap if overlap else "none"}')

errors: 76/154
  SUPPORTS -> NOT_ENOUGH_INFO: 16
  NOT_ENOUGH_INFO -> SUPPORTS: 14
  DISPUTED -> SUPPORTS: 13
  REFUTES -> DISPUTED: 8
  REFUTES -> NOT_ENOUGH_INFO: 7
  REFUTES -> SUPPORTS: 7
  SUPPORTS -> DISPUTED: 4
  NOT_ENOUGH_INFO -> DISPUTED: 3
  SUPPORTS -> REFUTES: 2
  NOT_ENOUGH_INFO -> REFUTES: 1
  DISPUTED -> NOT_ENOUGH_INFO: 1
ret_hit+cls_ok=44  ret_hit+cls_fail=29  ret_miss+cls_ok=34  ret_miss+cls_fail=47

claim: This means that the world is now 1C warmer than it was in pre-industrial times
true=SUPPORTS  pred=NOT_ENOUGH_INFO
overlap: {'evidence-889933', 'evidence-694262'}

claim: Greenland has only lost a tiny fraction of its ice mass
true=REFUTES  pred=DISPUTED
overlap: {'evidence-264761', 'evidence-52981'}

claim: CO2 limits won't cool the planet.
true=NOT_ENOUGH_INFO  pred=DISPUTED
overlap: none

claim: The actual data show high northern latitudes are warmer today than in 1940.
true=SUPPORTS  pred=NOT_ENOUGH_INFO
overlap: none

claim: CFCs contribute to global waerming

In [ ]:
if os.path.exists(os.path.join(ROBERTA_FINAL_DIR, 'config.json')):
    roberta_final = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_FINAL_DIR, num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID).to(DEVICE)
    print('final model loaded')
else:
    if os.path.exists(ALL_SAMPLES_PATH):
        with open(ALL_SAMPLES_PATH, 'rb') as f: all_samples = pickle.load(f)
    else:
        all_samples = train_samples + dev_samples
        with open(ALL_SAMPLES_PATH, 'wb') as f: pickle.dump(all_samples, f)

    all_label_dist = Counter(s['label'] for s in all_samples)
    all_total = sum(all_label_dist.values())
    all_weights = torch.tensor(
        [all_total / (n_classes * all_label_dist[i]) for i in range(n_classes)], dtype=torch.float32).to(DEVICE)
    final_criterion = LabelSmoothingCrossEntropy(smoothing=0.1, weight=all_weights)

    all_train_ds = TransformerDataset(all_samples, roberta_tokenizer)
    all_train_loader = DataLoader(all_train_ds, batch_size=16, shuffle=True)

    roberta_final = AutoModelForSequenceClassification.from_pretrained(
        'roberta-base', num_labels=4, id2label=ID2LABEL, label2id=LABEL2ID).to(DEVICE)

    FINAL_EPOCHS = 12
    opt_final = AdamW(roberta_final.parameters(), lr=1e-5, weight_decay=0.05)
    sched_final = get_linear_schedule_with_warmup(opt_final,
        num_warmup_steps=int(0.1 * len(all_train_loader) * FINAL_EPOCHS),
        num_training_steps=len(all_train_loader) * FINAL_EPOCHS)

    for epoch in range(FINAL_EPOCHS):
        tr_loss, tr_acc = train_epoch(roberta_final, all_train_loader, opt_final, sched_final, final_criterion)
        print(f'Epoch {epoch+1}/{FINAL_EPOCHS} | loss={tr_loss:.4f} acc={tr_acc:.4f}')

    roberta_final.save_pretrained(ROBERTA_FINAL_DIR)
    roberta_tokenizer.save_pretrained(ROBERTA_FINAL_DIR)

primary_final_model = roberta_final
primary_final_tokenizer = roberta_tokenizer

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

final model loaded


In [ ]:
dev_pred_path = DATA_DIR + 'dev-claims-predictions.json'
with open(dev_pred_path, 'w') as f:
    json.dump(dev_predictions, f, indent=2)
print(f'dev predictions saved')

test_pred_ckpt = CKPT_DIR + 'test_predictions.json'
test_pred_path = DATA_DIR + 'test-output.json'
if os.path.exists(test_pred_ckpt):
    with open(test_pred_ckpt) as f:
        test_predictions = json.load(f)
else:
    test_predictions = run_pipeline(test_claims, primary_final_model, primary_final_tokenizer)
    with open(test_pred_ckpt, 'w') as f:
        json.dump(test_predictions, f, indent=2)

with open(test_pred_path, 'w') as f:
    json.dump(test_predictions, f, indent=2)
print(f'test output written to {test_pred_path}')

sample_id = list(test_predictions.keys())[0]
print(json.dumps({sample_id: test_predictions[sample_id]}, indent=2))

dev predictions saved
test output written to /content/drive/MyDrive/COMP90042/test-output.json
{
  "claim-2967": {
    "claim_text": "The contribution of waste heat to the global climate is 0.028 W/m2.",
    "claim_label": "DISPUTED",
    "evidences": [
      "evidence-308923",
      "evidence-43606",
      "evidence-632574",
      "evidence-963856",
      "evidence-905191"
    ]
  }
}


## Object Oriented Programming codes here

*These are added here but used throughout my actual notebook and would break the code if I were to move them all down here unless I run this first then go back up... so here is all the OOP code down here and it won't break anything it'll just run like normal*

In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1, weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.weight = weight

    def forward(self, logits, targets):
        n_cls = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)
        with torch.no_grad():
            smooth = torch.full_like(log_probs, self.smoothing / (n_cls - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        loss = -(smooth * log_probs).sum(dim=-1)
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()

In [ ]:
class TransformerDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=256):
        self.labels = [s['label'] for s in samples]
        self.encodings = tokenizer([s['text'] for s in samples],
            truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return ({k: v[idx] for k, v in self.encodings.items()},
                torch.tensor(self.labels[idx], dtype=torch.long))


class BiLSTMDataset(Dataset):
    def __init__(self, samples, encoder):
        self.labels = [s['label'] for s in samples]
        self.embs = encoder.encode([s['text'] for s in samples], batch_size=32,
            show_progress_bar=True, convert_to_numpy=True, device=sbert_encode_device)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (torch.tensor(self.embs[idx], dtype=torch.float32),
                torch.tensor(self.labels[idx], dtype=torch.long))

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, num_layers=2, num_classes=4, dropout=0.4, chunk_size=64):
        super().__init__()
        self.chunk_size = chunk_size
        self.seq_len = input_dim // chunk_size
        self.bilstm = nn.LSTM(input_size=chunk_size, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        B = x.size(0)
        x = x.view(B, self.seq_len, self.chunk_size)
        out, _ = self.bilstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        ctx = (w * out).sum(dim=1)
        return self.classifier(self.drop(ctx))